# Introduction

Presidio is an open source library from Microsoft for PII analysis (detection) and anonymization.  
Once we detect the information to anonymize using functions from `presidio_analyzer`, we can apply functions from `presidio_anonymizer` to perform data anonymization.

## What is PII?
PII stands for **P**ersonally **I**dentifiable **I**nformation.
PII is any information that can identify a specific person, either on its own or when combined with other data.

Common examples of PII:

* Direct identifiers
  * Full name
  * Home address
  * Email address
  * Phone number
  * Social Security number
  * Passport or driver’s license number
   
* Indirect identifiers (still PII when combined):
  * Date of birth
  * IP address
  * Device IDs
  * Employment details
  * Location data
  * Online usernames (in many contexts)




# Install packages

First, we install `presidio_analyzer`.

In [1]:
!pip install -qq presidio_analyzer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.7/128.7 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.9/105.9 kB 6.8 MB/s eta 0:00:00


Then, we install `presidio_anonymizer`.

In [2]:
!pip install -qq presidio_anonymizer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 48.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pyopenssl 25.3.0 requires cryptography<47,>=45.0.7, but you have cryptography 44.0.3 which is incompatible.
pydrive2 1.21.3 requires cryptography<44, but you have cryptography 44.0.3 which is incompatible.
pydrive2 1.21.3 requires pyOpenSSL<=24.2.1,>=19.1.0, but you have pyopenssl 25.3.0 which is incompatible.


We also need to download `en_core_web_lg` library for spacy.

In [3]:
!python -m spacy download en_core_web_lg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 4.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


# Import packages

In [4]:
from presidio_analyzer import AnalyzerEngine, PatternRecognizer
from presidio_anonymizer import AnonymizerEngine
import json
from pprint import pprint

# Test analyzer with a simple text

In [5]:
text_to_anonymize = "His name is Mr. Jones and his phone number is 212-555-5555"

In [6]:
analyzer = AnalyzerEngine()
analyzer_results = analyzer.analyze(text=text_to_anonymize, language='en')

pprint(analyzer_results)

[type: PERSON, start: 16, end: 21, score: 0.85,
 type: PHONE_NUMBER, start: 46, end: 58, score: 0.75]


# Test anonymizer

With the results of the previous analysis, let's perform now anonymization on the same initial text.

In [7]:
# Initialize the engine:
engine = AnonymizerEngine()

# Invoke the anonymize function with the text analyzer results coming from presidio-analyzer
# Operators to get the anonymization output:
result = engine.anonymize(
    text=text_to_anonymize, analyzer_results=analyzer_results
)

print("De-identified text")
print(result.text)

De-identified text
His name is Mr. <PERSON> and his phone number is <PHONE_NUMBER>


# Add more items to anonymization

Let's add a list of titles to anonymize.

In [8]:
titles_list = [
    "Sir",
    "Ma'am",
    "Madam",
    "Mr.",
    "Mrs.",
    "Ms.",
    "Miss",
    "Dr.",
    "Professor",
    "Prof.",
    "san"
]

We initialize a `title_recognizer` object by passing to the `PatternRecognizer` the `deny_list` defined above.

In [9]:
from presidio_analyzer import PatternRecognizer

titles_recognizer = PatternRecognizer(supported_entity="TITLE", deny_list=titles_list)

Let's add `title_recognizer` to the analyzer (using `registry.add_recognizer`).

In [10]:
from presidio_analyzer import AnalyzerEngine

analyzer = AnalyzerEngine()
analyzer.registry.add_recognizer(titles_recognizer)

Now we can test again the initial text.

In [11]:
analyzer_results = analyzer.analyze(text=text_to_anonymize, language='en')

pprint(analyzer_results)

[type: TITLE, start: 12, end: 15, score: 1.0,
 type: PERSON, start: 16, end: 21, score: 0.85,
 type: PHONE_NUMBER, start: 46, end: 58, score: 0.75]


Let's anonymize now the text, using the extended analyze result (with the registered pattern recognizer for texts).

In [12]:
result = engine.anonymize(
    text=text_to_anonymize, analyzer_results=analyzer_results
)

print("De-identified text")
print(result.text)

De-identified text
His name is <TITLE> <PERSON> and his phone number is <PHONE_NUMBER>


# Extend tests

Let's check now both analyzer and anonymizer on few more texts.

In [13]:
texts = ["Hi support, I can't log in! My account username is 'johndoe88'. Every time I try, it says 'invalid credentials'.\
        Please reset my password. You can reach me at (555) 123-4567 or johnd@example.com",
         "My name is Soong Hil Num and I am a British citizen. My phone number is 094123456 and my email address is soong@adobe.com",
        "Last time Mr. Lang Lang Li visited us was on board of Ms. Fae Li from Tokyo",
        "Nakamura-san did one last attempt to learn the Chinese filmography from Miss Anne Admiral and use its input to talk with Dr. Yound Li",
        "Material science expert Dr. Li Young Sun met our specialist Prof. Franie Kuang to evaluate the feasibility of the approach.",
        "This text does not contain PII information and therefore should not be marked as so."]

In [14]:
for text in texts:
    print("\nInitial text:")
    print(text)
    analyzer_results = analyzer.analyze(text=text, language='en')
    print("Analysis result:")
    pprint(analyzer_results)
    result = engine.anonymize(text=text, analyzer_results=analyzer_results)
    print("De-identified text")
    print(result.text)


Initial text:
Hi support, I can't log in! My account username is 'johndoe88'. Every time I try, it says 'invalid credentials'.        Please reset my password. You can reach me at (555) 123-4567 or johnd@example.com
Analysis result:
[type: EMAIL_ADDRESS, start: 184, end: 201, score: 1.0,
 type: URL, start: 190, end: 201, score: 0.5,
 type: PHONE_NUMBER, start: 166, end: 180, score: 0.4]
De-identified text
Hi support, I can't log in! My account username is 'johndoe88'. Every time I try, it says 'invalid credentials'.        Please reset my password. You can reach me at <PHONE_NUMBER> or <EMAIL_ADDRESS>

Initial text:
My name is Soong Hil Num and I am a British citizen. My phone number is 094123456 and my email address is soong@adobe.com
Analysis result:
[type: EMAIL_ADDRESS, start: 106, end: 121, score: 1.0,
 type: PERSON, start: 11, end: 24, score: 0.85,
 type: NRP, start: 36, end: 43, score: 0.85,
 type: DATE_TIME, start: 72, end: 81, score: 0.85,
 type: PHONE_NUMBER, start: 72, end:

# Tests with selected targets

Let's repeat the tests, by specifying what categories we want to subject to analysis. We will keep only:
- PERSON
- TITLE
- EMAIL_ADDRESS
- PHONE_NUMBER

In [15]:
for text in texts:
    print("\nInitial text:")
    print(text)
    analyzer_results = analyzer.analyze(text=text, language='en',
            entities = ["PERSON", "TITLE", "EMAIL_ADDRESS", "PHONE_NUMBER"])
    print("Analysis result:")
    pprint(analyzer_results)
    result = engine.anonymize(text=text, analyzer_results=analyzer_results)
    print("De-identified text")
    print(result.text)


Initial text:
Hi support, I can't log in! My account username is 'johndoe88'. Every time I try, it says 'invalid credentials'.        Please reset my password. You can reach me at (555) 123-4567 or johnd@example.com
Analysis result:
[type: EMAIL_ADDRESS, start: 184, end: 201, score: 1.0,
 type: PHONE_NUMBER, start: 166, end: 180, score: 0.4]
De-identified text
Hi support, I can't log in! My account username is 'johndoe88'. Every time I try, it says 'invalid credentials'.        Please reset my password. You can reach me at <PHONE_NUMBER> or <EMAIL_ADDRESS>

Initial text:
My name is Soong Hil Num and I am a British citizen. My phone number is 094123456 and my email address is soong@adobe.com
Analysis result:
[type: EMAIL_ADDRESS, start: 106, end: 121, score: 1.0,
 type: PERSON, start: 11, end: 24, score: 0.85,
 type: PHONE_NUMBER, start: 72, end: 81, score: 0.75]
De-identified text
My name is <PERSON> and I am a British citizen. My phone number is <PHONE_NUMBER> and my email address is

# Observations and final remarks


The set of tests conducted in the first run using `Presidio` on several text examples, with various names, titles, email addresses, resulted in just few issues:
* In the first text example, **'johndoe88'** was not identified as a user name (might be out of scope for `Presidio` since it is not technically a person name).
* In the second example, **094123456** was identified as:
```
 type: DATE_TIME, start: 72, end: 81, score: 0.85,
 type: PHONE_NUMBER, start: 72, end: 81, score: 0.75,
 type: URL, start: 112, end: 121, score: 0.5,
 type: US_SSN, start: 72, end: 81, score: 0.05,
 type: US_BANK_NUMBER, start: 72, end: 81, score: 0.05,
 type: US_PASSPORT, start: 72, end: 81, score: 0.05,
 type: US_DRIVER_LICENSE, start: 72, end: 81, score: 0.01

The correct result is `type: PHONE_NUMBER, start: 72, end: 81, score: 0.75,` but because this one has lower score (0.75) that the 1st one (DATE_TIME, with 0.85), was not inserted as type during anonymization.
```  
* In the 4th example, the name (Nakamura) with title (san) **Nakamura-san**, because **Nakamura-san** was identified as a name in its entirety (**0-12&&), the title (detected upon analysis in positions **9-12**) was not shown as a token in the anonymized version, since the title was completely superposed by the last part of the name compound.



When we run the second set of tests, with the same texts, but by selecting only a reduced number of entities, we correctly detected and anonymized all the items corresponding to these entities, the rest being ignored. 

